# D124 — Shopping Cart Unit Testing

Build a shopping cart one behavior at a time and test each behavior immediately.

We use Python's built-in `unittest`; no installation is required.

## Business rules

- A `CartItem` has `id`, `name`, `qty`, `unit_price`, and calculated `amount`.
- A `Cart` holds zero or more items.
- `items_count` is the sum of all item quantities.
- `amount` is the sum of all item amounts.
- Every cart change must refresh both totals.

## Complete CartItem class

`amount` is calculated from the current quantity and unit price.

In [ ]:
class CartItem:
    def __init__(self, item_id, name, qty, unit_price):
        if qty <= 0:
            raise ValueError("Quantity must be positive")
        if unit_price < 0:
            raise ValueError("Unit price cannot be negative")
        self.id, self.name = item_id, name
        self.qty, self.unit_price = qty, unit_price

    @property
    def amount(self):
        return self.qty * self.unit_price

## Test CartItem first

Test the object before using it inside a cart.

In [ ]:
import unittest

class TestCartItem(unittest.TestCase):
    def test_amount_is_qty_times_price(self):
        item = CartItem(1, "Keyboard", 2, 1500)
        self.assertEqual(item.amount, 3000)

    def test_zero_quantity_is_rejected(self):
        with self.assertRaises(ValueError):
            CartItem(1, "Keyboard", 0, 1500)

## Build Cart step by step

For teaching in a notebook, each new function is attached to `Cart` after it is written.
This lets us test one operation before adding the next one.

A complete copy-ready class is provided at the end.

## Step 1 — initialize the cart

A new cart starts with no items, zero count, and zero amount.

In [ ]:
class Cart:
    def __init__(self):
        self.initialize_cart()

    def initialize_cart(self):
        self.items = {}
        self.items_count = 0
        self.amount = 0

In [ ]:
class TestCartInitialization(unittest.TestCase):
    def test_new_cart_is_empty(self):
        cart = Cart()
        self.assertEqual(cart.items, {})
        self.assertEqual(cart.items_count, 0)
        self.assertEqual(cart.amount, 0)

## Step 2 — common total calculation

All cart-changing methods will call this one business-logic method.

In [ ]:
def calculate_cart_total(self):
    self.items_count = sum(item.qty for item in self.items.values())
    self.amount = sum(item.amount for item in self.items.values())
    return self.amount

Cart.calculate_cart_total = calculate_cart_total

## Step 3 — add an item

A new ID creates a cart entry. Adding the same ID increases its quantity.

In [ ]:
def add_item_to_cart(self, item):
    if not isinstance(item, CartItem):
        raise TypeError("item must be a CartItem")
    if item.id in self.items:
        self.items[item.id].qty += item.qty
    else:
        self.items[item.id] = item
    self.calculate_cart_total()

Cart.add_item_to_cart = add_item_to_cart

## Test setup

`setUp` runs **before every test method**.
Each test receives a fresh cart and fresh items.

In [ ]:
class TestAddItem(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.keyboard = CartItem(1, "Keyboard", 2, 1500)

    def test_add_new_item_updates_state(self):
        self.cart.add_item_to_cart(self.keyboard)
        self.assertEqual(self.cart.items_count, 2)
        self.assertEqual(self.cart.amount, 3000)
        self.assertIn(1, self.cart.items)

In [ ]:
def test_same_item_increases_quantity(self):
    self.cart.add_item_to_cart(self.keyboard)
    self.cart.add_item_to_cart(CartItem(1, "Keyboard", 1, 1500))
    self.assertEqual(self.cart.items[1].qty, 3)
    self.assertEqual(self.cart.items_count, 3)
    self.assertEqual(self.cart.amount, 4500)

TestAddItem.test_same_item_increases_quantity = test_same_item_increases_quantity

## Step 4 — remove an item

Remove by item ID. An unknown ID is treated as an error.

In [ ]:
def remove_item_from_cart(self, item_id):
    if item_id not in self.items:
        raise KeyError("Item not found")
    removed_item = self.items.pop(item_id)
    self.calculate_cart_total()
    return removed_item

Cart.remove_item_from_cart = remove_item_from_cart

In [ ]:
class TestRemoveItem(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))

    def test_remove_existing_item(self):
        removed = self.cart.remove_item_from_cart(1)
        self.assertEqual(removed.name, "Keyboard")
        self.assertEqual(self.cart.items_count, 0)
        self.assertEqual(self.cart.amount, 0)

In [ ]:
def test_unknown_item_raises_key_error(self):
    with self.assertRaises(KeyError):
        self.cart.remove_item_from_cart(99)

TestRemoveItem.test_unknown_item_raises_key_error = (
    test_unknown_item_raises_key_error
)

## Step 5 — empty the cart

Emptying clears all items and refreshes both state values.

In [ ]:
def empty_cart(self):
    self.items.clear()
    self.calculate_cart_total()

Cart.empty_cart = empty_cart

In [ ]:
class TestEmptyCart(unittest.TestCase):
    def test_empty_cart_resets_all_state(self):
        cart = Cart()
        cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))
        cart.add_item_to_cart(CartItem(2, "Mouse", 1, 500))
        cart.empty_cart()
        self.assertEqual(cart.items, {})
        self.assertEqual(cart.items_count, 0)
        self.assertEqual(cart.amount, 0)

## Step 6 — update quantity or price

At least one new value is required. Values must follow the item rules.

In [ ]:
def update_cart(self, item_id, qty=None, unit_price=None):
    if item_id not in self.items:
        raise KeyError("Item not found")
    if qty is None and unit_price is None:
        raise ValueError("Provide quantity or unit price")
    if qty is not None and qty <= 0:
        raise ValueError("Quantity must be positive")
    if unit_price is not None and unit_price < 0:
        raise ValueError("Unit price cannot be negative")
    item = self.items[item_id]
    item.qty = qty if qty is not None else item.qty
    item.unit_price = unit_price if unit_price is not None else item.unit_price
    self.calculate_cart_total()

Cart.update_cart = update_cart

In [ ]:
class TestUpdateCart(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))

    def test_update_quantity(self):
        self.cart.update_cart(1, qty=3)
        self.assertEqual(self.cart.items_count, 3)
        self.assertEqual(self.cart.amount, 4500)

    def test_update_price(self):
        self.cart.update_cart(1, unit_price=1200)
        self.assertEqual(self.cart.amount, 2400)

In [ ]:
def test_update_quantity_and_price(self):
    self.cart.update_cart(1, qty=1, unit_price=1000)
    self.assertEqual(self.cart.items[1].amount, 1000)
    self.assertEqual(self.cart.items_count, 1)
    self.assertEqual(self.cart.amount, 1000)

TestUpdateCart.test_update_quantity_and_price = (
    test_update_quantity_and_price
)

In [ ]:
def test_invalid_updates_are_rejected(self):
    with self.assertRaises(ValueError):
        self.cart.update_cart(1, qty=0)
    with self.assertRaises(ValueError):
        self.cart.update_cart(1, unit_price=-1)
    with self.assertRaises(ValueError):
        self.cart.update_cart(1)

TestUpdateCart.test_invalid_updates_are_rejected = (
    test_invalid_updates_are_rejected
)

## Complete the error-path coverage

Add the remaining invalid-input and reset cases before building the suite.

In [ ]:
def test_negative_price_is_rejected(self):
    with self.assertRaises(ValueError):
        CartItem(1, "Keyboard", 1, -1)

TestCartItem.test_negative_price_is_rejected = (
    test_negative_price_is_rejected
)

In [ ]:
def test_add_rejects_non_cart_item(self):
    with self.assertRaises(TypeError):
        self.cart.add_item_to_cart("Keyboard")

TestAddItem.test_add_rejects_non_cart_item = (
    test_add_rejects_non_cart_item
)

In [ ]:
def test_initialize_resets_existing_cart(self):
    self.cart.add_item_to_cart(self.keyboard)
    self.cart.initialize_cart()
    self.assertEqual(self.cart.items, {})
    self.assertEqual(self.cart.items_count, 0)
    self.assertEqual(self.cart.amount, 0)

TestAddItem.test_initialize_resets_existing_cart = (
    test_initialize_resets_existing_cart
)

In [ ]:
def test_update_unknown_item_raises_key_error(self):
    with self.assertRaises(KeyError):
        self.cart.update_cart(99, qty=2)

TestUpdateCart.test_update_unknown_item_raises_key_error = (
    test_update_unknown_item_raises_key_error
)

## `setUp` and `tearDown`

- `setUp()` runs before each test.
- `tearDown()` runs after each test, even if the test fails.
- Use them for repeated preparation and cleanup.
- In-memory objects usually need little cleanup.

In [ ]:
class TestLifecycle(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()

    def tearDown(self):
        self.cart.empty_cart()

    def test_cart_starts_empty(self):
        self.assertEqual(self.cart.items_count, 0)

## What is a test suite?

A **test case** is one test method.
A **test class** groups related test cases.
A **test suite** groups test classes or selected tests for one run.

In [ ]:
test_classes = [
    TestCartItem, TestCartInitialization, TestAddItem,
    TestRemoveItem, TestEmptyCart, TestUpdateCart, TestLifecycle
]

suite = unittest.TestSuite()
for test_class in test_classes:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(test_class))

## Run the complete test suite

`verbosity=2` displays every test name and result.

In [ ]:
result = unittest.TextTestRunner(verbosity=2).run(suite)

assert result.wasSuccessful()
print("Tests run:", result.testsRun)

## Complete functional coverage in this lesson

- item amount and invalid item quantity
- cart initialization
- add a new item and add an existing item
- remove an item and reject an unknown ID
- empty the cart
- update quantity, price, or both
- reject invalid updates
- verify `items_count` and `amount` after every change

## Copy-ready final Cart class

This is the same behavior consolidated into a normal class definition.

In [ ]:
class Cart:
    def __init__(self):
        self.initialize_cart()

    def initialize_cart(self):
        self.items, self.items_count, self.amount = {}, 0, 0

    def calculate_cart_total(self):
        self.items_count = sum(x.qty for x in self.items.values())
        self.amount = sum(x.amount for x in self.items.values())
        return self.amount

    def add_item_to_cart(self, item):
        if not isinstance(item, CartItem):
            raise TypeError("item must be a CartItem")
        if item.id in self.items:
            self.items[item.id].qty += item.qty
        else:
            self.items[item.id] = item
        self.calculate_cart_total()

    def remove_item_from_cart(self, item_id):
        if item_id not in self.items:
            raise KeyError("Item not found")
        removed = self.items.pop(item_id)
        self.calculate_cart_total()
        return removed

    def empty_cart(self):
        self.items.clear()
        self.calculate_cart_total()

    def update_cart(self, item_id, qty=None, unit_price=None):
        if item_id not in self.items:
            raise KeyError("Item not found")
        if qty is None and unit_price is None:
            raise ValueError("Provide quantity or unit price")
        if qty is not None and qty <= 0:
            raise ValueError("Quantity must be positive")
        if unit_price is not None and unit_price < 0:
            raise ValueError("Unit price cannot be negative")
        item = self.items[item_id]
        item.qty = qty if qty is not None else item.qty
        item.unit_price = unit_price if unit_price is not None else item.unit_price
        self.calculate_cart_total()

## Final smoke test

Confirm that the consolidated class behaves like the step-by-step version.

In [ ]:
cart = Cart()
cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))
cart.add_item_to_cart(CartItem(2, "Mouse", 1, 500))
assert cart.items_count == 3
assert cart.amount == 3500

cart.update_cart(1, qty=1)
assert cart.items_count == 2
assert cart.amount == 2000

## Key lessons

- Add one behavior, then test it.
- Keep shared state calculation in one method.
- Verify both the direct result and changed object state.
- Use `setUp` for fresh test data and `tearDown` for cleanup.
- Use a suite to run related test classes together.
- Run the full suite after every change.